## Imports and Environment

In [27]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings, PromptTemplate, get_response_synthesizer, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.query_engine import RetrieverQueryEngine, TransformQueryEngine
from llama_index.core.retrievers import BaseRetriever, VectorIndexRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, util
from PIL import Image
from pdf2image import convert_from_path

import pandas as pd, re, ast, textwrap
from datasets import Dataset

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory, LlamaIndexEmbeddingsWrapper
from ragas.metrics import answer_relevancy, faithfulness, context_recall, context_precision
from ragas import evaluate

from dotenv import load_dotenv, find_dotenv
from typing import List, Any

import torch
import os
import re
import csv
import requests
import gc
import multiprocessing as mp
import concurrent.futures
from tqdm import tqdm

# --- For Azure ML Sandpit environment ---

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# # Change the current working directory to the project root
os.chdir(project_root_path)

dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# --- For Local Development environment ---
# dotenv_path = find_dotenv()

# Load the .env file from the path that was found.
load_dotenv(dotenv_path=dotenv_path)

# Get the project root from the directory where the .env file was found.
project_root = os.path.dirname(dotenv_path)

# Get the relative directory name from the environment variable
relative_data_dir = os.getenv("VECTOR_DATASET_DIR")

# Create the full, absolute path by joining the project root with the relative name.
data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

print(f"✅ Project root automatically determined as: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

## Initialise Models

In [2]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    max_new_tokens=1024,
    model_kwargs={"token": hf_token, "dtype": torch.bfloat16},
    generate_kwargs={
        "temperature": 0.1,
        "repetition_penalty": 1.2,
        "do_sample": True,
    }
)

print("Meta-Llama-3.1-8B-Instruct initialized for conversational responses.")

# Initialise Nanonets OCR for document parsing
ocr_model_name = "nanonets/Nanonets-OCR-s"
try:
    ocr_processor = AutoProcessor.from_pretrained(ocr_model_name)
    ocr_model = AutoModelForImageTextToText.from_pretrained(
        ocr_model_name,
        device_map="auto",
        dtype=torch.bfloat16 # Corrected argument
    )
    print(f"✅ Nanonets OCR model ('{ocr_model_name}') loaded successfully.")
except Exception as e:
    print(f"❌ Failed to load Nanonets OCR model. Error: {e}")

# HyDE prompt template
my_hyde_prompt = PromptTemplate(
    "Drawing from the style of a Singapore Police Force annual crime report, write a "
    "concise, data-driven paragraph that directly answers the user's question. "
    "Focus on using specific crime-related keywords, percentages, and years. Do not invent statistics.\\n\\n"
    "Question: {context_str}\\n\\n"
    "Report Excerpt:"
)

# Initialize HyDE query transformation
hyde_transform = HyDEQueryTransform(
    llm=llm,
    include_original=True,
    hyde_prompt=my_hyde_prompt
)

print("HyDE Query Transform initialized.")

# Initialize Re-ranker LLM
reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-v2-m3",
    top_n=5  # Number of nodes to return after re-ranking
)

## Initialise Vector Embeddings

In [3]:
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(
    model_name="intfloat/multilingual-e5-large",
    token=hf_token  # Pass the token here
)

print("Global settings configured with intfloat/multilingual-e5-large.")

## Vector Store Data Ingestion

In [ ]:
all_documents = []
print("--- Starting data ingestion using the Nanonets model directly ---")

NANONETS_PROMPT = """Extract the text from the above document as if you were reading it naturally. Return the tables in html format. Return the equations in LaTeX representation. If there is an image in the document and image caption is not present, add a small description of the image inside the <img></img> tag; otherwise, add the image caption inside <img></img>. Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. Prefer using ☐ and ☑ for check boxes."""

def batch_ocr_on_pages(images, ocr_model, ocr_processor, page_batch_size=8):
    """
    Processes a list of page images in batches to maximize GPU utilization.
    
    Args:
        images (list): A list of PIL Image objects.
        ocr_model: The loaded Nanonets model.
        ocr_processor: The loaded Nanonets processor.
        page_batch_size (int): The number of pages to process in a single model call.
    
    Returns:
        list: A list of extracted text strings, one for each page.
    """
    all_texts = []
    # Create batches of page images
    for i in tqdm(range(0, len(images), page_batch_size), desc="OCR on Pages"):
        batch_images = images[i:i + page_batch_size]
        
        # Prepare the model inputs for the batch
        # The Nanonets processor is designed to handle a list of images
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": NANONETS_PROMPT},
                ],
            },
        ]
        text = ocr_processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = ocr_processor(text=text, images=batch_images, padding=True, return_tensors="pt").to(ocr_model.device)

        # Generate text for the entire batch at once
        output_ids = ocr_model.generate(**inputs, max_new_tokens=4096, do_sample=False)
        
        # Decode the results for the batch
        generated_ids = [output_id[len(input_id):] for input_id, output_id in zip(inputs.input_ids, output_ids)]
        batch_outputs = ocr_processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        
        all_texts.extend(batch_outputs)

        # Clean up to manage memory
        del batch_images, inputs, output_ids
        gc.collect()
        
    return all_texts

# --- Main Ingestion Loop ---
all_documents = []
print("--- Starting data ingestion with batched page OCR ---")

PAGE_BATCH_SIZE = 8

files_to_process = [f for f in os.listdir(data_directory) if f.lower().endswith('.pdf')]
num_total_files = len(files_to_process)
print(f"Found {num_total_files} PDF files to process.")

# Process files one by one (for memory stability)
for filename in tqdm(files_to_process, desc="Processing Files"):
    filepath = os.path.join(data_directory, filename)
    
    try:
        images_from_pdf = convert_from_path(filepath, dpi=150)
        
        page_texts = batch_ocr_on_pages(images_from_pdf, ocr_model, ocr_processor, page_batch_size=PAGE_BATCH_SIZE)
        
        full_text_content = "\n".join(page_texts)
        
        if full_text_content.strip():
            document = Document(text=full_text_content, metadata={"filepath": filepath})
            all_documents.append(document)
        else:
            print(f"- WARNING: No content was extracted from {filename}")

        # Clean up memory-intensive image list
        del images_from_pdf
        gc.collect()

    except Exception as e:
        print(f"\n- FAILED to process {filename}. Error: {e}")
        continue

documents = all_documents
print("\n--- Ingestion complete ---")
print(f"Successfully loaded {len(documents)} documents.")

## Chunking

In [5]:
node_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
pipeline = IngestionPipeline(transformations=[node_parser])
nodes = pipeline.run(documents=documents)

print(f"Total nodes created with custom chunking: {len(nodes)}")
print(f"Chunk size: {node_parser.chunk_size}, Chunk overlap: {node_parser.chunk_overlap}")

## Initialise Retrievers

In [6]:
# Instantiate VectorStoreIndex
vector_index = VectorStoreIndex(nodes)
semantic_retriever = VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=5
)

# Instantiate the BM25Retriever
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=5
)

print("Vector and BM25 retrievers ready (each top_k=5).")

## Define Ensemble Retriever

In [7]:
class DualPathEnsembleRetriever(BaseRetriever):
    """Runs BM25 and semantic retrieval in parallel, applies a weight to BM25, and merges results."""
    
    def __init__(self, bm25: BM25Retriever, semantic: VectorIndexRetriever, bm25_weight: float = 2.0):
        """
        Initializes the retriever.
        
        Args:
            bm25 (BM25Retriever): The BM25 retriever instance.
            semantic (VectorIndexRetriever): The semantic (vector) retriever instance.
            bm25_weight (float): A multiplier to boost the scores from BM25. Defaults to 2.0.
        """
        self._bm25 = bm25
        self._semantic = semantic
        self._bm25_weight = bm25_weight
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """The core retrieval logic."""
        # Run both retrieval paths in parallel
        bm25_hits = self._bm25.retrieve(query_bundle)
        sem_hits = self._semantic.retrieve(query_bundle)

        # Apply the weight to the BM25 scores
        for node_with_score in bm25_hits:
            node_with_score.score *= self._bm25_weight

        # Concatenate and de-duplicate by node_id, keeping the one with the best score
        combined = {}
        for hit in bm25_hits + sem_hits:
            nid = hit.node.node_id
            if nid not in combined or hit.score > combined[nid].score:
                combined[nid] = hit
        
        # Sort the final results by score in descending order
        sorted_combined = sorted(list(combined.values()), key=lambda x: x.score, reverse=True)

        return sorted_combined

# Control the weight directly
dual_retriever = DualPathEnsembleRetriever(
    bm25_retriever, 
    semantic_retriever, 
    # bm25_weight=2.0 # Multiply scores from bm_25 by 2.0
)

print("Dual-path ensemble retriever initialized with BM25 weight factor.")

## Retrieve Answer from Datastore

In [8]:
custom_prompt = PromptTemplate(
    "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
    "Do not invent or add anything not present in the context. Do not repeat the question. "
    "Do not answer any other questions. Output exactly one sentence and then stop.\\n\\n"
    "Context:\\n"
    "---------------------\\n"
    "{context_str}\\n"
    "---------------------\\n\\n"
    "User's Question: {query_str}\\n"
    "Answer: "
)

response_synthesizer = get_response_synthesizer(
    llm=llm,
    text_qa_template=custom_prompt
)

# Base engine uses the dual-path retriever
ensemble_query_engine = RetrieverQueryEngine(
    retriever=dual_retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[reranker],  # rerank the merged set
)

# Apply HyDE first, then dual-path retrieval, then rerank, then LLM
advanced_query_engine = TransformQueryEngine(
    ensemble_query_engine,
    query_transform=hyde_transform
)

query_text = "What percentage of reported crimes in 2020 were scams?"
response = ensemble_query_engine.query(query_text) # Without HyDE
# response = advanced_query_engine.query(query_text) # With HyDE
print("--- Retrieved Source Nodes ---")
for node in response.source_nodes:
    print(f"Node Score: {node.score}")
    print(node.get_content())
    print("-" * 20)
print(str(response))

## Debugging to check HyDe Document

In [ ]:
# original_bundle = QueryBundle(query_text)

# # Manually execute the transform to see what it produces
# transformed_bundle = hyde_transform(original_bundle)

# # The `embedding_strs` field now holds the text that will be used for vector search.
# # With `include_original=True`, this list contains both the new hypothetical document
# # and the original query.
# hypothetical_doc = transformed_bundle.embedding_strs[0]

# print("--- Hypothetical Document Generated by HyDE for Retrieval ---")
# print(textwrap.fill(hypothetical_doc, width=90))
# print("-------------------------------------------------------------")

# # The original query string is preserved for the final answer synthesis step.
# print(f"\\nQuery used for final answer synthesis: '{transformed_bundle.query_str}'")


## Transform Extracted Answer To Be Conversational

In [9]:
# Normalize the extracted answer
# Eg "19,966." -> "19,966"
extracted_answer = str(response)
answer_core = extracted_answer.strip()
answer_core = re.sub(r"\.\s*$", "", answer_core).strip()

if not answer_core:
    print("Sorry, I couldn’t extract an answer from the context.")
else:
    REPHRASE_PROMPT = (
       "Rephrase the following information into a single, simple, conversational sentence. "
        "Your response must contain the Extracted Answer exactly as it is provided. "
        "Do not add any extra information, justifications, or explanations. "
        "Do not repeat the question."
        "End your response with a single period.\\n\\n"
        
        f"Extracted answer: {answer_core}\\n"
        f"Question: {query_text}\\n"
        "Conversational Answer:"
)

    # Generate a short rephrased sentence to avoid loops
    rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=32)

    # Collapse whitespace
    s = " ".join(str(rephrase_raw).strip().split())

    # Remove a potential leading echo of the bare answer (e.g., "19,966. The total ...")
    if answer_core:
        s = re.sub(rf"^\s*{re.escape(answer_core)}\.\s*", "", s).strip()

    # Ensure exactly one sentence
    idx = s.find(".")
    s = (s[: idx + 1] if idx != -1 else s + ".").strip()

    # Enforce inclusion of the extracted answer exactly once
    # If the model dropped the value, fall back to the minimal guaranteed version
    if answer_core not in s:
        s = f"{answer_core}."

    print(s)

## Benchmarking

In [ ]:
# --- Configuration ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")

BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(3)adv_vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load Data ---
print(f"Loading benchmark data from: {BENCHMARK_FILE_PATH}")
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']
# Slice to first 3 rows for quick test
# benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- Unified Generation and Context Selection ---
def get_response_and_top_context(question: str, retriever):
    """
    Retrieves context, generates a response, and returns the single top-ranked
    context chunk for clear reporting.
    """
    # Retrieve the top N nodes which are sorted by relevance
    retrieved_nodes = retriever.retrieve(question)
    
    # De-duplicate and clean all retrieved chunks for the LLM
    seen = set()
    all_cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            all_cleaned_chunks.append(content)
            seen.add(content)
    
    # The top-ranked context is simply the first one in the list
    top_context_chunk = all_cleaned_chunks if all_cleaned_chunks else ""
    
    # Combine ALL chunks into a single context pool for the LLM
    context_for_llm = "\\n\\n---\\n\\n".join(all_cleaned_chunks)

    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\\n\\n"
        "Context:\\n"
        "---------------------\\n"
        "{context_str}\\n"
        "---------------------\\n\\n"
        "User's Question: {query_str}\\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=context_for_llm, query_str=question)
    
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response to a single sentence
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts and parts else s
    
    # Return the prediction, all chunks for Ragas, and the top chunk for the CSV
    return prediction, all_cleaned_chunks, top_context_chunk


# --- Generate Predictions using the base_pipeline's logic ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("\nGenerating predictions using the base pipeline's logic...")
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # Get the response and context using the unified function
    prediction, all_chunks, top_chunk = get_response_and_top_context(question, advanced_query_engine)
    
    # Store results in the DataFrame matching the base pipeline's format
    benchmark_df.at[i, 'response'] = prediction
    benchmark_df.at[i, 'retrieved_contexts'] = top_chunk
    
    # Collect data for Ragas evaluation
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction)
    ragas_data["contexts"].append(all_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = LlamaIndexEmbeddingsWrapper(Settings.embed_model)
print(judge_embeddings)

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Desired Column Order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")

## Benchmarking Base Without HyDE

In [ ]:
print("Creating the Query Engine for benchmarking...")
semantic_query_engine = RetrieverQueryEngine.from_args(
    retriever=semantic_retriever,
    response_synthesizer=get_response_synthesizer(),
    node_postprocessors=[reranker]
)

# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(3base+reranker)base_vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']

# --- CONFIG: Set the number of rows to test ---
# Slice to first 3 rows for quick test
# benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# --- Unified Generation and Context Pinpointing Logic ---
def get_response_and_top_context(question: str, retriever):
    """
    Retrieves context, generates a response, and returns the single top-ranked
    context chunk for clear reporting.
    """
    # Retrieve the top N nodes which are sorted by relevance
    query_bundle = QueryBundle(question)
    retrieved_nodes = retriever.retrieve(query_bundle)
    
    # De-duplicate and clean all retrieved chunks for the LLM
    seen = set()
    all_cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            all_cleaned_chunks.append(content)
            seen.add(content)
    
    # The top-ranked context is simply the first one in the list.
    top_context_chunk = all_cleaned_chunks if all_cleaned_chunks else ""
    
    # Combine ALL chunks into a single context pool for the LLM
    context_for_llm = "\\n\\n---\\n\\n".join(all_cleaned_chunks)

    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\\n\\n"
        "Context:\\n"
        "---------------------\\n"
        "{context_str}\\n"
        "---------------------\\n\\n"
        "User's Question: {query_str}\\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=context_for_llm, query_str=question)
    
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response to a single sentence
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts and parts else s
    
    # Return the prediction, all chunks for Ragas, and the top chunk for the CSV
    return prediction, all_cleaned_chunks, top_context_chunk


# --- Generate Predictions using the base_pipeline's logic ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("\\nGenerating predictions using the base pipeline's logic...")
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # Get the response and context using the unified function
    prediction, all_chunks, top_chunk = get_response_and_top_context(question, semantic_query_engine)
    
    # Store results in the DataFrame, matching the base pipeline's format
    benchmark_df.at[i, 'response'] = prediction
    benchmark_df.at[i, 'retrieved_contexts'] = top_chunk
    
    # Collect data for Ragas evaluation
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction)
    ragas_data["contexts"].append(all_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="text-embedding-ada-002")

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Ensure the desired column order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")

## Benchmarking Base Without Reranker

In [ ]:
print("Creating the Query Engine for benchmarking...")
semantic_query_engine = RetrieverQueryEngine.from_args(
    retriever=semantic_retriever,
    response_synthesizer=get_response_synthesizer(),
)

base_without_reranker = TransformQueryEngine(
    semantic_query_engine,
    query_transform=hyde_transform
)
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(3base+hyde)base_vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']

# --- CONFIG: Set the number of rows to test ---
# Slice to first 3 rows for quick test
# benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# --- Unified Generation and Context Pinpointing Logic ---
def get_response_and_top_context(question: str, retriever):
    """
    Retrieves context, generates a response, and returns the single top-ranked
    context chunk for clear reporting.
    """
    # Retrieve the top N nodes which are sorted by relevance
    query_bundle = QueryBundle(question)
    retrieved_nodes = retriever.retrieve(query_bundle)
    
    # De-duplicate and clean all retrieved chunks for the LLM
    seen = set()
    all_cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            all_cleaned_chunks.append(content)
            seen.add(content)
    
    # The top-ranked context is simply the first one in the list.
    top_context_chunk = all_cleaned_chunks if all_cleaned_chunks else ""
    
    # Combine ALL chunks into a single context pool for the LLM
    context_for_llm = "\\n\\n---\\n\\n".join(all_cleaned_chunks)

    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\\n\\n"
        "Context:\\n"
        "---------------------\\n"
        "{context_str}\\n"
        "---------------------\\n\\n"
        "User's Question: {query_str}\\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=context_for_llm, query_str=question)
    
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response to a single sentence
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts and parts else s
    
    # Return the prediction, all chunks for Ragas, and the top chunk for the CSV
    return prediction, all_cleaned_chunks, top_context_chunk


# --- Generate Predictions using the base_pipeline's logic ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("\\nGenerating predictions using the base pipeline's logic...")
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # Get the response and context using the unified function
    prediction, all_chunks, top_chunk = get_response_and_top_context(question, base_without_reranker)
    
    # Store results in the DataFrame, matching the base pipeline's format
    benchmark_df.at[i, 'response'] = prediction
    benchmark_df.at[i, 'retrieved_contexts'] = top_chunk
    
    # Collect data for Ragas evaluation
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction)
    ragas_data["contexts"].append(all_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="text-embedding-ada-002")

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Desired column order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")

## Benchmarking Advanced without HyDE

In [ ]:
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(3base+reranker)vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']

# --- CONFIG: Set the number of rows to test ---
# Slice to first 3 rows for quick test
# benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# --- Unified Generation and Context Pinpointing Logic ---
def get_response_and_top_context(question: str, retriever):
    """
    Retrieves context, generates a response, and returns the single top-ranked
    context chunk for clear reporting.
    """
    # Retrieve the top N nodes which are sorted by relevance
    query_bundle = QueryBundle(question)
    retrieved_nodes = retriever.retrieve(query_bundle)
    
    # De-duplicate and clean all retrieved chunks for the LLM
    seen = set()
    all_cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            all_cleaned_chunks.append(content)
            seen.add(content)
    
    # The top-ranked context is simply the first one in the list.
    top_context_chunk = all_cleaned_chunks if all_cleaned_chunks else ""
    
    # Combine ALL chunks into a single context pool for the LLM
    context_for_llm = "\\n\\n---\\n\\n".join(all_cleaned_chunks)

    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\\n\\n"
        "Context:\\n"
        "---------------------\\n"
        "{context_str}\\n"
        "---------------------\\n\\n"
        "User's Question: {query_str}\\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=context_for_llm, query_str=question)
    
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response to a single sentence
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts and parts else s
    
    # Return the prediction, all chunks for Ragas, and the top chunk for the CSV
    return prediction, all_cleaned_chunks, top_context_chunk


# --- Generate Predictions using the base_pipeline's logic ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("\\nGenerating predictions using the base pipeline's logic...")
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # Get the response and context using the unified function
    prediction, all_chunks, top_chunk = get_response_and_top_context(question, ensemble_query_engine)
    
    # Store results in the DataFrame, matching the base pipeline's format
    benchmark_df.at[i, 'response'] = prediction
    benchmark_df.at[i, 'retrieved_contexts'] = top_chunk
    
    # Collect data for Ragas evaluation
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction) # Use the direct prediction
    ragas_data["contexts"].append(all_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="intfloat/multilingual-e5-large")

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Desired column order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")


## Benchmarking Advance without Reranker

In [ ]:
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(3base+HyDE)vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']

# Slice to first 3 rows for quick test
# benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# --- Unified Generation and Context Pinpointing Logic ---
def get_response_and_top_context(question: str, retriever):
    """
    Retrieves context, generates a response, and returns the single top-ranked
    context chunk for clear reporting.
    """
    # Retrieve the top N nodes which are sorted by relevance
    retrieved_nodes = retriever.retrieve(question)
    
    # De-duplicate and clean all retrieved chunks for the LLM
    seen = set()
    all_cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            all_cleaned_chunks.append(content)
            seen.add(content)
    
    # The top-ranked context is simply the first one in the list.
    top_context_chunk = all_cleaned_chunks if all_cleaned_chunks else ""
    
    # Combine ALL chunks into a single context pool for the LLM
    context_for_llm = "\\n\\n---\\n\\n".join(all_cleaned_chunks)

    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\\n\\n"
        "Context:\\n"
        "---------------------\\n"
        "{context_str}\\n"
        "---------------------\\n\\n"
        "User's Question: {query_str}\\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=context_for_llm, query_str=question)
    
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response to a single sentence
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts and parts else s
    
    # Return the prediction, all chunks for Ragas, and the top chunk for the CSV
    return prediction, all_cleaned_chunks, top_context_chunk

# --- Define no reranker engine
base_engine_for_hyde = RetrieverQueryEngine.from_args(
    retriever=dual_retriever
)

# Now, wrap the base engine with TransformQueryEngine to apply HyDE.
engine_without_reranker = TransformQueryEngine(
    base_engine_for_hyde,
    query_transform=hyde_transform
)

# --- Generate Predictions using the base_pipeline's logic ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("\\nGenerating predictions using the base pipeline's logic...")
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # Get the response and context using the unified function
    prediction, all_chunks, top_chunk = get_response_and_top_context(question, engine_without_reranker)
    
    # Store results in the DataFrame, matching the base pipeline's format
    benchmark_df.at[i, 'response'] = prediction
    benchmark_df.at[i, 'retrieved_contexts'] = top_chunk
    
    # Collect data for Ragas evaluation
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction) # Use the direct prediction
    ragas_data["contexts"].append(all_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="intfloat/multilingual-e5-large")

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Desired column order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")
